# Notebook 01: Tokenizer 对比 — sentencepiece vs HuggingFace tokenizers

本 notebook 对比两种 BPE tokenizer 实现：
1. **sentencepiece** (from-scratch 版 ClearMind 使用)
2. **HuggingFace tokenizers** (本项目 ClearMind-HF 使用)

对比维度：训练 API、encode/decode、特殊 token 管理、chat template 支持。

In [ ]:
import sys, os, tempfile, json
sys.path.insert(0, os.path.abspath(".."))

# 准备测试语料
corpus_dir = tempfile.mkdtemp()
corpus_path = os.path.join(corpus_dir, "corpus.txt")

texts = [
    "深度学习是机器学习的一个子领域，它利用多层神经网络来学习数据的层次化表示。",
    "Transformer架构于2017年被提出，彻底改变了自然语言处理领域的研究方向。",
    "Deep learning is a subset of machine learning that uses neural networks.",
    "The Transformer architecture has revolutionized natural language processing.",
    "注意力机制允许模型在处理序列数据时，关注输入中最相关的部分。",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Pre-trained language models acquire powerful language understanding capabilities.",
    "预训练语言模型通过在大量文本数据上进行无监督学习，获得了强大的语言理解能力。",
    "反向传播算法通过链式法则计算损失函数对每个参数的梯度，是训练神经网络的核心方法。",
    "Backpropagation uses the chain rule to compute gradients of the loss function.",
]

with open(corpus_path, "w", encoding="utf-8") as f:
    for _ in range(50):  # 重复以获得足够训练数据
        for t in texts:
            f.write(t + "\n")

print(f"Corpus: {corpus_path}")
print(f"Lines: {50 * len(texts)}")

## 1. 训练 API 对比

### sentencepiece (from-scratch 版)
```python
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix="tokenizer",
    vocab_size=500,
    model_type="bpe",
    character_coverage=0.9995,
    user_defined_symbols=["<sep>", "<mask>"],
)
sp = spm.SentencePieceProcessor(model_file="tokenizer.model")
```
- 输出单个 `.model` 文件（二进制 protobuf）
- 特殊 token 需通过 `user_defined_symbols` 手动指定
- 无标准 save/load 接口，需要手动管理文件

### HuggingFace tokenizers (本项目)
下面实际演示：

In [ ]:
# HF tokenizers 训练 — 使用 ClearMindTokenizer 工厂类
from src.data.tokenizer import ClearMindTokenizer

tokenizer = ClearMindTokenizer.train(corpus_path=corpus_path, vocab_size=500)

print(f"Type: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: unk={tokenizer.unk_token}, bos={tokenizer.bos_token}, eos={tokenizer.eos_token}, pad={tokenizer.pad_token}")
print(f"\n底层实现: tokenizers.Tokenizer (Rust) -> PreTrainedTokenizerFast (Python wrapper)")
print(f"输出文件: tokenizer.json + tokenizer_config.json (JSON 格式，可读性强)")

## 2. Encode/Decode 对比

### sentencepiece
```python
ids = sp.encode("Hello, world!")       # [1, 234, 56, ...]
text = sp.decode(ids)                   # "Hello, world!"
tokens = sp.encode("Hello", out_type=str)  # ['▁He', 'llo']
```
- 使用 `▁` (U+2581) 表示空格前缀
- encode/decode 不自动添加 BOS/EOS

### HuggingFace tokenizers
- 使用 ByteLevel pre-tokenizer（字节级别）
- encode 自动添加 BOS/EOS（通过 TemplateProcessing）
- 完整的 `__call__` 接口返回 `input_ids` + `attention_mask`

In [ ]:
# Encode/Decode 演示
test_texts = [
    "Hello, world!",
    "深度学习是机器学习的一个子领域。",
    "Transformer architecture",
]

for text in test_texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids, skip_special_tokens=True)
    tokens = tokenizer.tokenize(text)
    print(f"Text:    '{text}'")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print(f"Decoded: '{decoded}'")
    print(f"BOS={ids[0]==tokenizer.bos_token_id}, EOS={ids[-1]==tokenizer.eos_token_id}")
    print()

## 3. Chat Template 对比

### sentencepiece (from-scratch 版)
需要手动拼接对话格式：
```python
def format_chat(messages):
    result = ""
    for msg in messages:
        if msg["role"] == "user":
            result += f"Human: {msg['content']}\n"
        elif msg["role"] == "assistant":
            result += f"Assistant: {msg['content']}"
    return result
```

### HuggingFace tokenizers
内置 Jinja2 chat template，通过 `apply_chat_template()` 自动格式化：

In [ ]:
# Chat template 演示
messages = [
    {"role": "user", "content": "什么是深度学习？"},
    {"role": "assistant", "content": "深度学习是机器学习的一个分支。"},
    {"role": "user", "content": "它和传统机器学习有什么区别？"},
]

# 不添加生成提示
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print("=== Without generation prompt ===")
print(formatted)

# 添加生成提示（推理时使用）
formatted_gen = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("\n=== With generation prompt ===")
print(formatted_gen)

# Chat template 内容
print(f"\n=== Jinja2 Template ===")
print(tokenizer.chat_template)

## 4. Save/Load 对比

### sentencepiece
```python
# 保存: 训练时直接输出 .model 文件
# 加载:
sp = spm.SentencePieceProcessor(model_file="tokenizer.model")
```
- 单个二进制文件，无法直接查看内容
- 无标准的 `save_pretrained` / `from_pretrained` 接口

### HuggingFace tokenizers
- JSON 格式文件，人类可读
- 标准 `save_pretrained` / `from_pretrained` 接口
- 与 HuggingFace Hub 无缝集成

In [ ]:
# Save/Load 演示
save_dir = os.path.join(corpus_dir, "saved_tokenizer")
tokenizer.save_pretrained(save_dir)

print("Saved files:")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# 加载并验证一致性
loaded = ClearMindTokenizer.load(save_dir)
test = "Deep learning and Transformer"
assert tokenizer.encode(test) == loaded.encode(test)
print(f"\nLoad verification: encode('{test}') matches!")

## 5. 总结对比

| 特性 | sentencepiece | HF tokenizers |
|------|-------------|---------------|
| **底层实现** | C++ | Rust |
| **分词算法** | Unigram/BPE | BPE (ByteLevel) |
| **空格处理** | `▁` 前缀 | ByteLevel (无信息丢失) |
| **输出格式** | 二进制 `.model` | JSON (`tokenizer.json`) |
| **特殊 token** | 手动管理 | `special_tokens_map` 自动管理 |
| **Chat template** | 需手动实现 | Jinja2 内置支持 |
| **Save/Load** | 手动文件管理 | `save_pretrained` / `from_pretrained` |
| **HF 生态集成** | 需要 adapter | 原生支持 |
| **训练速度** | 快 | 更快 (Rust 并行) |

**结论**: HF tokenizers 提供更好的生态集成和标准化接口，适合与 HuggingFace Transformers 配合使用。sentencepiece 更轻量，适合独立部署的场景。